# KAIROS VERIFIER — Replay

**Menu → Runtime → Run all.** No command line knowledge needed.

This notebook downloads the published verifier script, checks its identity against
the SHA-256 recorded in the public repository, runs it inside the sealed
execution envelope, and shows the verdict below — `SEALED_RUNTIME_MATCH` /
`DETERMINISM_PASS` if everything matches.

Repository: https://github.com/KAIROSSYSTEMSCH/KAIROS-VERIFIER

**Note — Colab environment.** The next step may print red "ERROR: pip's dependency resolver..." text. This concerns packages preinstalled by Colab for other purposes (`google-colab`, `numba`), not this script. The correct versions are installed regardless — continue to the next step.

In [ ]:
#@title Step 1 — Install pinned dependencies
!pip install --quiet numpy==2.2.6 pandas==2.3.3 xgboost==3.2.0

In [ ]:
#@title Step 2 — Download and authenticate the script
import urllib.request
import hashlib

SCRIPT_URL = "https://raw.githubusercontent.com/KAIROSSYSTEMSCH/KAIROS-VERIFIER/main/determinism_ladder.py"
EXPECTED_SHA256 = "e2f6cd4b4e67e98d19adbd76e9e8c12eae0276cca3755b324e55f3b966525be4"

urllib.request.urlretrieve(SCRIPT_URL, "determinism_ladder.py")

with open("determinism_ladder.py", "rb") as f:
    obtained = hashlib.sha256(f.read()).hexdigest()

print("SHA-256 obtenu  :", obtained)
print("SHA-256 attendu :", EXPECTED_SHA256)
print()

assert obtained == EXPECTED_SHA256, (
    "MISMATCH — le script telecharge ne correspond pas a l'empreinte publiee. "
    "Ne pas continuer. Ouvrez une issue sur le depot."
)
print("Identite du script confirmee - vous executez le fichier reellement publie.")

In [ ]:
#@title Step 3 — Run inside the sealed envelope
import subprocess
import os
import sys

env = os.environ.copy()
env.update({
    "OPENBLAS_CORETYPE": "Haswell",
    "OMP_NUM_THREADS": "1",
    "MKL_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
    "PYTHONHASHSEED": "0",
})

result = subprocess.run(
    [sys.executable, "determinism_ladder.py"],
    env=env, capture_output=True, text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("--- stderr ---")
    print(result.stderr)

---
**Reading the result.** `SEALED_RUNTIME_MATCH` confirms this Colab session matches
the declared envelope (pinned library versions, BLAS kernel, thread caps,
hash seed). `DETERMINISM_PASS` confirms the twelve measurements were stable
across three repeated passes within this session. Compare the twelve hashes
printed above to [`EXPECTED_HASHES.md`](https://github.com/KAIROSSYSTEMSCH/KAIROS-VERIFIER/blob/main/EXPECTED_HASHES.md)
in the repository.

This session is ephemeral and not preserved by Kairos. The output above is your
own record of this replay.